In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
cwd = os.getcwd()
data_dir = os.path.join(cwd, 'data')
out_dir = os.path.join(cwd, 'output')

In [2]:
# Loading raw data 
raw_data_df = pd.read_csv(os.path.join(data_dir, 'MDS-UPDRS_Part_III_24Jan2024.csv'))

# Getting relevant columns
# PATNO: Patient number
# EVENT_ID: Event identifier (e.g. v0, v1...)
# INFODT: Date of the visit
# PDTRTMNT: Taking Parkinson's medication (1: Yes, 0: No)
# PDSTATE: Medication state (ON or OFF)
# [23:-6]: MDS-UPDRSI-III items
# NP3TOT: MDS-UPDRS-III total score
# NHY: Hoehn and Yahr stage

columns_of_interest = ['PATNO', 'EVENT_ID', 'INFODT', 'PDTRTMNT', 'PDSTATE', 
                       'NP3TOT', 'NHY'] + raw_data_df.columns.tolist()[23:-6]
raw_data_df = raw_data_df[columns_of_interest]
print(f"Total entries: {len(raw_data_df)} \nTotal unique patients: {raw_data_df['PATNO'].nunique()}")
raw_data_df


Total entries: 25104 
Total unique patients: 3382


,PATNO,EVENT_ID,INFODT,PDTRTMNT,PDSTATE,NP3TOT,NHY,NP3SPCH,NP3FACXP,NP3RIGN,...,NP3PTRMR,NP3PTRML,NP3KTRMR,NP3KTRML,NP3RTARU,NP3RTALU,NP3RTARL,NP3RTALL,NP3RTALJ,NP3RTCON
0,3000,BL,02/2011,NaN,NaN,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3000,V04,03/2012,NaN,NaN,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3000,V06,02/2013,NaN,NaN,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3000,V08,03/2014,NaN,NaN,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3000,V10,03/2015,NaN,NaN,19.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25099,293823,BL,11/2023,0.0,NaN,12.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
25100,293825,BL,01/2024,0.0,NaN,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25101,293846,BL,01/2024,0.0,NaN,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25102,294119,BL,12/2023,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
# Cleaning dataframe

# Remove rows with missing values in any row
df = raw_data_df.dropna()
print(f"Total entries after dropping rows with missing values: {len(df)} \nTotal unique patients: {df['PATNO'].nunique()}")

# Remove "on" medication state
df = df[df['PDSTATE'] != 'ON']
print(f"Total entries after removing 'ON' medication state: {len(df)} \nTotal unique patients: {df['PATNO'].nunique()}")

# Transforming INFODT to datetime format
df['INFODT'] = pd.to_datetime(df['INFODT'], format='%m/%Y')

# transforming EVENT_ID to months since baseline
visit_dict ={
    'BL' : 0, 'V01' : 3, 'V02' : 6, 'V03' : 9, 'V04' : 12,
    'V05' : 18, 'V06' : 24, 'V07' : 30, 'V08' : 36, 'V09' : 42,
    'V10' : 48, 'V11' : 54, 'V12' : 60, 'V13' : 72, 'V14' : 84,
    'V15' : 96, 'V16' : 108, 'V17' : 120, 'V18' : 132, 'V19' : 144,
    'V20' : 156}
df['MONTH'] = df['EVENT_ID'].map(visit_dict)
# removing rows with missing values in MONTH column
df = df.dropna(subset=['MONTH'])
print(f"Total entries after removing rows with missing MONTH values: {len(df)} \nTotal unique patients: {df['PATNO'].nunique()}")

Total entries after dropping rows with missing values: 9502 
Total unique patients: 970
Total entries after removing 'ON' medication state: 3685 
Total unique patients: 856
Total entries after removing rows with missing MONTH values: 3653 
Total unique patients: 854


In [4]:
# Adding MDS-UPDRS-III sub-scores
 # 'NP3SPCH','NP3FACXP', 
### Rigidity ###
 # 'NP3RIGN','NP3RIGRU', 'NP3RIGLU','NP3RIGRL', 'NP3RIGLL'
### Bradykinesia ###
 # 'NP3FTAPR', 'NP3FTAPL','NP3HMOVR', 'NP3HMOVL','NP3PRSPR', 'NP3PRSPL', 
 # 'NP3TTAPR', 'NP3TTAPL','NP3LGAGR', 'NP3LGAGL','NP3BRADY'
### Postural instability and gait disorder ###
 #  'NP3RISNG', 'NP3GAIT', 'NP3FRZGT', 'NP3PSTBL', 'NP3POSTR'
### Tremor ###
 # 'NP3PTRMR', 'NP3PTRML', 'NP3KTRMR','NP3KTRML',
 # 'NP3RTARU','NP3RTALU','NP3RTARL','NP3RTALL','NP3RTALJ','NP3RTCON']

df['RIGIDITY'] = df['NP3RIGN'] + df['NP3RIGRU'] + df['NP3RIGLU'] + df['NP3RIGRL'] + df['NP3RIGLL']
df['BRADY'] = df['NP3FTAPR'] + df['NP3FTAPL'] + df['NP3HMOVR'] + df['NP3HMOVL'] + df['NP3PRSPR'] + df['NP3PRSPL'] + df['NP3TTAPR'] + df['NP3TTAPL'] + df['NP3LGAGR'] + df['NP3LGAGL'] + df['NP3BRADY']
df['PIGD'] = df['NP3RISNG'] + df['NP3GAIT'] + df['NP3FRZGT'] + df['NP3PSTBL'] + df['NP3POSTR']
df['TREMOR'] = df['NP3PTRMR'] + df['NP3PTRML'] + df['NP3KTRMR'] + df['NP3KTRML'] + df['NP3RTARU'] + df['NP3RTALU'] + df['NP3RTARL'] + df['NP3RTALL'] + df['NP3RTALJ'] + df['NP3RTCON']

In [5]:
# saving master df
df.to_csv(os.path.join(data_dir, 'master_df.csv'), index=False)

In [83]:
# Plotting for visualizing results
%matplotlib qt
# Checking sample size for each stage
early_df = df[df['NHY'] <= 2]
middle_df = df[(df['NHY'] > 2) & (df['NHY'] <= 4)]
late_df = df[df['NHY'] > 4]

# early_df = df[df['NHY'] <= 2]
# middle_df = df[(df['NHY'] > 2)]
# late_df = df[df['NHY'] > 4]

print(f"Early stage entries: {len(early_df)} \nEarly stage unique patients: {early_df['PATNO'].nunique()}")
print(f"Middle stage entries: {len(middle_df)} \nMiddle stage unique patients: {middle_df['PATNO'].nunique()}")
print(f"Late stage entries: {len(late_df)} \nLate stage unique patients: {late_df['PATNO'].nunique()}")


for measure in ['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']:
    plt.figure(figsize=(10, 6))
    # plotting scatter + regression line for each stage separately
    plt.scatter(early_df['MONTH'], early_df[measure], label='Early Stage', alpha=0.5, color='b')
    m, b = np.polyfit(early_df['MONTH'], early_df[measure], 1)
    plt.plot(early_df['MONTH'], m*np.array(early_df['MONTH']) + b, color='b')

    plt.scatter(middle_df['MONTH'], middle_df[measure], label='Middle Stage', alpha=0.5, color='g')
    m, b = np.polyfit(middle_df['MONTH'], middle_df[measure], 1)
    plt.plot(middle_df['MONTH'], m*np.array(middle_df['MONTH']) + b, color='g')

    plt.scatter(late_df['MONTH'], late_df[measure], label='Late Stage', alpha=0.5, color='r')
    m, b = np.polyfit(late_df['MONTH'], late_df[measure], 1)
    plt.plot(late_df['MONTH'], m*np.array(late_df['MONTH']) + b, color='r')

    plt.xlabel('Month')
    plt.ylabel(f'{measure} Score')
    plt.title('Disease Progression Over Time')
    plt.legend()
    plt.show()


Early stage entries: 3259 
Early stage unique patients: 818
Middle stage entries: 374 
Middle stage unique patients: 205
Late stage entries: 20 
Late stage unique patients: 17


In [ ]:
# PLotting for visualizing results (regreesion with 95%CI)
for measure in ['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']:
    plt.figure(figsize=(10, 6))

    # Early
    sns.regplot(
        data=early_df, x='MONTH', y=measure,
        scatter=False,  # removes scatter
        ci=95,
        label='Early Stage',
        color='b',
        line_kws={'linewidth': 2}
    )

    # Middle
    sns.regplot(
        data=middle_df, x='MONTH', y=measure,
        scatter=False,
        ci=95,
        label='Middle Stage',
        color='g',
        line_kws={'linewidth': 2}
    )

    # Late
    sns.regplot(
        data=late_df, x='MONTH', y=measure,
        scatter=False,
        ci=95,
        label='Late Stage',
        color='r',
        line_kws={'linewidth': 2}
    )

    plt.xlabel('Month')
    plt.ylabel(f'{measure} Score')
    plt.title(f'{measure} Progression Over Time')
    plt.legend()
    plt.show()

In [84]:
def multiple_dvs_lmm(df, dvs, ivs = ' ~ MONTH', 
                     random_intercept='PATNO', random_slope='~MONTH', std_dv = False):
    """
    Fit linear mixed models (LMMs) for multiple dependent variables (DVs).

    Parameters:
    df (pd.DataFrame): DataFrame containing the data for modeling.
    dvs (list of str): List of dependent variable column names to model.
    ivs (str): Fixed effects formula (default includes Occasion, Age, and Sex).
    random_intercept (str): Column name to use for random intercept grouping (e.g., participant ID).
    random_slope (str): Random slope formula (default is '~Occasion').

    Returns:
    list: List of fitted LMM model results (MixedLMResults objects).
    """
    mdfs = []
    scaler = StandardScaler()

    for dv in dvs:

        # Standardize the dependent variable if specified
        if std_dv:
            standardized_col = dv + "_z"
            df[standardized_col] = scaler.fit_transform(df[[dv]])
            formula = standardized_col + ivs
        else:
            formula = dv + ivs # Construct the full formula: e.g., 'Measure ~ Occasion + Age + Sex'
        
        md = mixedlm(formula, data=df, groups=df[random_intercept], re_formula=random_slope)
        mdf = md.fit(method='lbfgs')
        mdfs.append(mdf)

    return mdfs

def univariate_lmm_dataframe(dvs, mdfs, output=None, save=False):
    """
    Create a summary DataFrame of parameter estimates and p-values for each DV's LMM.

    Parameters:
    dvs (list of str): List of dependent variable names corresponding to the models.
    mdfs (list): List of fitted LMM results.
    output (str): File path to save the output Excel file (if save=True).
    save (bool): Whether to save the summary DataFrame to an Excel file.

    Returns:
    pd.DataFrame: Multi-index DataFrame summarizing parameters and p-values per DV.
    """
    columns  = []
    ivs = mdfs[0].pvalues.dropna().index # Assume all models have the same IVs
    for iv in ivs:
        columns.append((iv, 'param')) # Parameter estimate
        columns.append((iv,'pval')) # p-value
        columns.append((iv, 'pval_fdr')) # FDR-corrected p-value
    
    df = pd.DataFrame(index=dvs, columns=pd.MultiIndex.from_tuples(columns))
    for dv, mdf in zip(dvs, mdfs):
        for iv in ivs:
            df.loc[dv, (iv, 'param')] = round(mdf.params[iv],3)
            df.loc[dv, (iv, 'pval')] = round(mdf.pvalues[iv],3)

        # Apply BH-FDR correction per IV across DVs
    for iv in ivs:
        # Convert to numeric, force None -> NaN
        pvals = pd.to_numeric(df[(iv, 'pval')], errors='coerce')

        # Mask valid p-values
        valid_mask = pvals.notna()

        # Initialize corrected p-values with NaN
        pvals_fdr = np.full(len(pvals), np.nan)

        # Apply FDR only to valid p-values
        if valid_mask.sum() > 0:
            _, pvals_fdr_valid, _, _ = multipletests(
                pvals[valid_mask].values,
                method='fdr_bh'
            )
            pvals_fdr[valid_mask] = pvals_fdr_valid

        # Assign back to DataFrame
        df[(iv, 'pval_fdr')] = np.round(pvals_fdr, 3)

        # Round raw p-values last
        df[(iv, 'pval')] = pvals.round(3)
    
    if save:
        df.to_excel(output)
    return df

In [ ]:
# Full cohort
mdfs = multiple_dvs_lmm(df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_all = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs, output=os.path.join(out_dir, 'lmm_all.xlsx'), 
                                      save=True)

# Early stage
mdfs_early = multiple_dvs_lmm(early_df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_early = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs_early, output=os.path.join(out_dir, 'lmm_early.xlsx'), 
                                      save=True)

# Middle stage
mdfs_middle = multiple_dvs_lmm(middle_df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_middle = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs_middle, output=os.path.join(out_dir, 'lmm_middle.xlsx'), 
                                      save=True)
# Late stage
mdfs_late = multiple_dvs_lmm(late_df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_late = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs_late, output=os.path.join(out_dir, 'lmm_late.xlsx'), 
                                      save=True)

c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 21.896146
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\stats